In [1]:
%load_ext autoreload
%autoreload 2
import import_before_profile

In [2]:
import jax
import jax.numpy as jnp

from import_me_for_testing import N_max_exps, aux_data, boundaries, dist, model
from qdots_qll.distributions import (
    Distribution,
    update_log_weights,
)
from qdots_qll.exp_design import RandExpDesignGAME
from qdots_qll.models.single_dot_weak_coupling_GAME import (
    SingleDotWeakCouplingGAME,
)
from data import Data
from experiments import ExperimentSingleDotWeakCouplingGAME
from qdots_qll.resamplers import LiuWestResampler, MetropolisSampler
from qdots_qll.smc import SMCUpdater, replace_single_datum, replace_single_exp

key = jax.random.key(0)

DEBUG:2024-09-03 17:53:53,996:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 17:53:53,997:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 17:53:53,999:jax._src.lru_cache:107: Cache hit for key: 'jit_convert_element_type-0581db0bf206ecfadd3967c7aa8a468505e540523fe3dc2b0e3e9ca72cd76bca'
DEBUG:2024-09-03 17:53:54,001:jax._src.compiler:98: Persistent compilation cache hit for 'jit_convert_element_type' with key 'jit_convert_element_type-0581db0bf206ecfadd3967c7aa8a468505e540523fe3dc2b0e3e9ca72cd76bca'
DEBUG:2024-09-03 17:53:54,109:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 17:53:54,109:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 17:53:54,111:jax._src.lru_cache:107: Cache hit fo

In [6]:
from qdots_qll.resamplers import MetropolisSampler

resampler = MetropolisSampler(boundaries, model)
exp_design = RandExpDesignGAME()

from qdots_qll.smc import SMCUpdater

In [7]:
key, subkey = jax.random.split(key)

smc = SMCUpdater(model, exp_design, resampler)

In [8]:
iteration = 0

smc.step(key, iteration, dist, aux_data)

DEBUG:2024-09-03 17:55:36,677:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 17:55:36,677:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 17:55:36,687:jax._src.lru_cache:107: Cache hit for key: 'jit_step-5f358db174c28835c670c1a7ac44ac97ae0e944a25c96030aeb556c048cdcb92'
DEBUG:2024-09-03 17:55:37,007:jax._src.compiler:98: Persistent compilation cache hit for 'jit_step' with key 'jit_step-5f358db174c28835c670c1a7ac44ac97ae0e944a25c96030aeb556c048cdcb92'


(Array((), dtype=key<fry>) overlaying:
 [1226047270 2014510724],
 Array(1, dtype=int32, weak_type=True),
 Distribution(
   no_rv=4,
   no_particles=500,
   particles_locations=f32[500,4],
   log_weights=f32[500],
   weights=f32[500],
   ESS=f32[]
 ),
 Data(
   experiment=ExperimentSingleDotWeakCouplingGAME(
     time=f32[10000,1],
     initial_state=i8[10000,1],
     measurement_basis=i8[10000,1]
   ),
   outcome=i8[10000,1]
 ))

In [15]:
def f_scan(list_of_args, x):
    carry = smc.step(*list_of_args)
    return carry, None

In [20]:
jax.lax.scan(f_scan, init=(key, iteration, dist, aux_data), length=4500)

DEBUG:2024-09-03 17:59:26,120:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 17:59:26,121:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 17:59:26,131:jax._src.lru_cache:104: Cache miss for key: 'jit_scan-ffe651b4b1b4c07beb88b070f32cf5029188a693280d7e4ae1b02b937e322bba'
DEBUG:2024-09-03 17:59:30,836:jax._src.compiler:704: 'jit_scan' took at least 0.00 seconds to compile (4.70s)


((Array((), dtype=key<fry>) overlaying:
  [3746310584  963981477],
  Array(4500, dtype=int32, weak_type=True),
  Distribution(
    no_rv=4,
    no_particles=500,
    particles_locations=f32[500,4],
    log_weights=f32[500],
    weights=f32[500],
    ESS=f32[]
  ),
  Data(
    experiment=ExperimentSingleDotWeakCouplingGAME(
      time=f32[10000,1],
      initial_state=i8[10000,1],
      measurement_basis=i8[10000,1]
    ),
    outcome=i8[10000,1]
  )),
 None)

In [16]:
f_scan((key, iteration, dist, aux_data), jnp.array([3]))

((Array((), dtype=key<fry>) overlaying:
  [1226047270 2014510724],
  Array(1, dtype=int32, weak_type=True),
  Distribution(
    no_rv=4,
    no_particles=500,
    particles_locations=f32[500,4],
    log_weights=f32[500],
    weights=f32[500],
    ESS=f32[]
  ),
  Data(
    experiment=ExperimentSingleDotWeakCouplingGAME(
      time=f32[10000,1],
      initial_state=i8[10000,1],
      measurement_basis=i8[10000,1]
    ),
    outcome=i8[10000,1]
  )),
 None)

In [5]:
# def _do_not_resample(key, dist: Distribution, *args, **kwargs):
#     return key, dist


# def _resample(key, dist, iteration, data, resampler, *args, **kwargs):
#     key, subkey = jax.random.split(key)
#     new_dist: Distribution = resampler.resample(
#         subkey=subkey,
#         index_data=iteration,
#         distribution=dist,
#         data=data,
#     )
#     return key, new_dist


# @jax.jit
# def f2(key, model, iteration, all_data, distribution, *args, **kwargs):
#     key, subkey = jax.random.split(key)
#     # experiment = e1
#     experiment = exp_design.generate_experiment(
#         model=model,
#         distribution=distribution,
#         data=all_data,
#         subkey=subkey,
#     )

#     outcome = model.measure_one_experiment(subkey, experiment)
#     datum = Data(experiment, outcome)

#     log_lkl = model.log_lkl_datum_multiple_particles(
#         distribution.particles_locations, datum
#     )

#     all_data = replace_single_datum(all_data, datum, iteration)

#     distribution: Distribution = update_log_weights(
#         dist=distribution, new_log_lkl=log_lkl
#     )

#     key, subkey = jax.random.split(key)
#     # distribution = resampler.resample(subkey, iteration, distribution, all_data)
#     key, distribution = jax.lax.cond(
#         distribution.check_resampling(),
#         _resample,
#         _do_not_resample,
#         *(key, distribution, iteration, all_data, resampler),
#     )

#     iteration = iteration + 1
#     return key, model, iteration, all_data, distribution